In [ ]:
from medmnist import PathMNIST
import torch
from torch import nn, utils, Generator
import torch.nn.functional as F
import torchvision
# from tqdm import trange, tqdm
from tqdm.notebook import trange, tqdm
import plotly.express as px
from typing import Sequence
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"
import plotly.colors as pc

import threading
from concurrent.futures import ThreadPoolExecutor

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0)) 
EPOCHS = 30

## Dataset Preparation

[MedMNIST](https://doi.org/10.1038/s41597-022-01721-8) is a collection of 18x standardized datasets for 2D and 3D biomedical image classification. 

It contains multiple size options: 28 (MNIST-Like), 64, 128, and 224.

In [ ]:
prepare_dataset = lambda x: PathMNIST(x, torchvision.transforms.ToTensor(), lambda x:x.squeeze(), download=True, root="/workspace", size=128)
ds_train, ds_val, ds_test = [prepare_dataset(x) for x in ['train', 'val', 'test']]

In [ ]:
# Downloading the dataset from zenodo can take a long time, to speed up the process:
# cd /var/tmp
# aria2c -x 16 -s 16 "https://zenodo.org/records/10519652/files/pathmnist_128.npz?download=1" -o pathmnist_128.npz

# Transforms only run when iterating over the dataset. Since we are processing the
# raw imgs and labels arrays, we need to apply these transforms ourselves.
def preload(dataset):
    X = torch.from_numpy(dataset.imgs).permute(0, 3, 1, 2).float() / 255.0
    y = torch.from_numpy(dataset.labels).squeeze()
    return utils.data.TensorDataset(X, y)

ds_train_t = preload(ds_train)
ds_val_t = preload(ds_val)
ds_test_t = preload(ds_test)

random_generator = Generator().manual_seed(42)
prepare_dataloader = lambda x: utils.data.DataLoader(
    x, 256, True,
    generator=random_generator,
    pin_memory=True,
)

dl_train, dl_val, dl_test = [prepare_dataloader(x) for x in [ds_train_t, ds_val_t, ds_test_t]]

classes, label_lookup = len(ds_train.info['label']), ds_train.info['label']

ds_train

In [ ]:
ex = next(iter(dl_train))
print(ex[0].shape, ex[1].shape)

img = torch.einsum("nchw->nhwc",ex[0][:10])
fig = px.imshow(img, binary_string=True, facet_col=0, facet_col_wrap=5, facet_col_spacing=0.01, facet_row_spacing=0.08)

item_map={f'{i}':label_lookup[str(key.item())] for i, key in enumerate(ex[1][:10])}
fig.for_each_annotation(lambda a: a.update(text=item_map[a.text.split("=")[1]]))

fig.update_layout(
        # title={"text": "N", "x": 0.5},
        height=600,
        width=1500,
        margin=dict(l=20, r=20, t=20, b=20),
    )

fig.write_image(f"figures/sample_images.svg")

fig.show()

## Model Definitions

In [ ]:
class NamedModule(nn.Module):
    def set_name(self, name:str):
        self.override_name = name
        return self

    def name(self) -> str:
        if hasattr(self, "override_name"):
            return self.override_name
        name_attr = getattr(self, "_name", None)
        if callable(name_attr):
            return name_attr() # type: ignore
        elif isinstance(name_attr, str):
            return name_attr
        else:
            raise NotImplementedError
    
class SoftmaxRegression(NamedModule):
    def __init__(self, outfeatures: int, bias = True):
        super().__init__()
        self.f = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(outfeatures, bias)
        )
    
    def name(self) -> str:
        return "softmax"

    def forward(self, x):
        return self.f(x)

class CNNBlock(NamedModule):
    def __init__(self, out_channels:int, conv_layers=4, skip_frequency=2, batch_norm=True):
        super().__init__()
        self.conv_layers = conv_layers
        self.features = nn.ModuleList()
        self.residuals = nn.ModuleList()
        self.skip_frequency = skip_frequency
        
        for i in range(conv_layers):
            self.features.append(nn.Sequential(nn.LazyConv2d(out_channels, 3, padding=1), *([nn.GroupNorm(1, out_channels)] if batch_norm else []), nn.ReLU()))
            if self._should_end_skip(i):
                self.residuals.append(nn.LazyConv2d(out_channels, 1))

    def _has_skips(self):
        return self.skip_frequency != 0

    def _should_start_skip(self, i):
        return self._has_skips() and (i%self.skip_frequency == 0 and (i+self.skip_frequency) <= self.conv_layers)
    
    def _should_end_skip(self, i):
        return self._has_skips() and ((i+1)%self.skip_frequency == 0)

    def forward(self, x):
        identity = 0
        for i, conv in enumerate(self.features):
            if self._should_start_skip(i):
                identity = self.residuals[i//self.skip_frequency](x)
            x = conv(x)
            if self._should_end_skip(i):
                x = x + identity
        return x

class CNN(NamedModule):
    r"""Construct VGG/ResNet hybrid CNN

    Core architecture of VGG remains the same, with the addition of skip
    connections within each block.

    Standard features of ResNet such as strides conv instead of max pool,
    global average pooling instead of FC layers and bottleneck blocks have
    not been included.
    
    Args:
        arch: List of tuples. Each tuple contains number of convolutions in layer, out channels of layer,
            residual frequency in layer. Set residual frequency to 0 to disable residuals.
        in_channels: Number of channels in the input image.
        classes: Number of logits.
        dropout_rate: Dropout used in the classifier layer.
    """

    def __init__(self, arch: list[tuple[int,int,int]], classes=10, dropout_rate=0.0, batch_norm=True):
        super().__init__()

        self.max_pool = nn.MaxPool2d(2)
        self.dropout_rate = dropout_rate
        self.features = nn.ModuleList()
        self.arch = arch

        layers = []
        for (num_convs, out_ch, residuals) in self.arch:
            layers.append(CNNBlock(out_ch, num_convs, residuals, batch_norm))
            layers.append(self.max_pool)

        self.features = nn.Sequential(*layers)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(512),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.LazyLinear(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.LazyLinear(classes),
        )

        # ResNet style classifier.
        # self.classifier = nn.Sequential(
        #     nn.AdaptiveAvgPool2d(1),
        #     nn.Flatten(),
        #     nn.LazyLinear(classes),
        # )

    def _name(self):
        convolutions = sum([k[0] for k in self.arch])
        skip_frequency = self.arch[0][2]
        n = f"cnn"
        n += f"-{convolutions}"
        if self.dropout_rate > 0:
            n += f"-d-{self.dropout_rate}"
        if skip_frequency > 0:
            n += f"-s-{skip_frequency}"
        return n

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
class Record():
    def __init__(self):
        self.store = {}
        self.epoch_store = {}
        self._lock = threading.Lock()

    def add(self, name, loss, logits, targets):
        with self._lock:
            self.store.setdefault(f"{name}-loss", []).append(loss)
            acc = (logits.argmax(1) == targets).float().mean().item()
            self.store.setdefault(f"{name}-acc", []).append(acc)

    def compute(self, prefix=""):
        with self._lock:
            for key, values in list(self.store.items()):
                if key.startswith(prefix):
                    self.epoch_store.setdefault(key, []).append(
                        sum(values) / len(values)
                    )
                    del self.store[key]

    def clear(self):
        with self._lock:
            self.store = {}
            self.epoch_store = {}

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import re


def plot_comparison(metrics: Record, title="Model Comparison", splits=("val", "train"), measures=("acc", "loss"), split_subplots=False, file_name=""):
    titles = {
        "acc": "Accuracy",
        "loss": "Loss",
        "val": "Validation",
        "train": "Train"
    }

    if split_subplots:
        rows = len(splits)
        cols = len(measures)
        subplot_titles = [f"{titles[s]} {titles[m]} vs Epoch" for s in splits for m in measures]
    else:
        rows = 1
        cols = len(measures)
        subplot_titles = [f"{titles[m]} vs Epoch" for m in measures]

    fig = make_subplots(rows=rows, cols=cols, subplot_titles=subplot_titles, horizontal_spacing=0.04, vertical_spacing=0.08)
    models = set(k.rsplit("-", 2)[0] for k in metrics.epoch_store.keys())
    colors = pc.qualitative.Plotly

    def natural_sort_key(s):
        return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', s)]

    trace_idx = 0
    shown = set()
    for name in sorted(models, key=natural_sort_key):
        if split_subplots:
            color = colors[trace_idx % len(colors)]
            trace_idx += 1
            for split in splits:
                for m_idx, measure in enumerate(measures):
                    key = f"{name}-{split}-{measure}"
                    values = metrics.epoch_store.get(key)
                    if values is None:
                        continue
                    epochs = list(range(len(values)))
                    show = name not in shown
                    shown.add(name)
                    fig.add_trace(
                        go.Scatter(x=epochs, y=values, name=name, legendgroup=name, showlegend=show, line=dict(color=color)),
                        row=splits.index(split) + 1, col=m_idx + 1
                    )
        else:
            for split in splits:
                color = colors[trace_idx % len(colors)]
                trace_idx += 1
                label = f"{name} ({split})" if len(splits) > 1 else name
                for m_idx, measure in enumerate(measures):
                    key = f"{name}-{split}-{measure}"
                    values = metrics.epoch_store.get(key)
                    if values is None:
                        continue
                    epochs = list(range(len(values)))
                    show = label not in shown
                    shown.add(label)
                    fig.add_trace(
                        go.Scatter(x=epochs, y=values, name=label, legendgroup=label, showlegend=show, line=dict(color=color)),
                        row=1, col=m_idx + 1
                    )

    fig.update_xaxes(title_text="Epoch")
    for m_idx, measure in enumerate(measures):
        for r in range(1, rows + 1):
            fig.update_yaxes(title_text=titles[measure], row=r, col=m_idx + 1)

    fig.update_layout(
        title={"text": title, "x": 0.5},
        showlegend=len(models) > 1 or len(splits) > 1,
        height=600 * rows,
        width=1500,
        margin=dict(l=20, r=20, t=50, b=20),
    )
    if file_name:
        fig.write_image(f"figures/{file_name}.svg")
    fig.show()
# plot_comparison(softmax_metrics, "Softmax", file_name="Softmax")
# plot_comparison(softmax_metrics, "Softmax", splits=("val","train"), measures=("acc","loss"))
# plot_comparison(metrics_batchnorm_off, "CNN Comparison Normalisation Off (Validation)", splits=("val",))
# plot_comparison(metrics_batchnorm_off, "CNN Comparison Normalisation Off (Train)", splits=("train",))
# plot_comparison(metrics_batchnorm_off, "CNN Comparison Normalisation Off", splits=("train","val"), split_subplots=True)

In [ ]:
def eval_model(model: NamedModule, metric_store: Record, device: torch.device, train_ds, val_ds, epochs=EPOCHS, lr=1e-3, position=0):
    dl_train = utils.data.DataLoader(train_ds, 256, shuffle=True, generator=Generator().manual_seed(42))
    dl_val = utils.data.DataLoader(val_ds, 256, generator=Generator().manual_seed(42))
    model.to(device)
    trainer = torch.optim.Adam(model.parameters(), lr=lr)
    loss = nn.CrossEntropyLoss()
    with trange(epochs, position=position, desc=model.name()) as t:
        for i in t:
            model.train()
            for X, y in dl_train:
                X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
                out = model(X)
                l = loss(out, y)
                trainer.zero_grad(set_to_none=True)
                l.backward()
                trainer.step()
                metric_store.add(f"{model.name()}-train", l.item(), out.detach(), y)

            model.eval()
            with torch.no_grad():
                for X, y in dl_val:
                    X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
                    out = model(X)
                    l = loss(out, y)
                    metric_store.add(f"{model.name()}-val", l.item(), out.detach(), y)

            metric_store.compute(prefix=model.name())
            t.set_description(f"{model.name()} ({max(metric_store.epoch_store[f'{model.name()}-val-acc'])*100:.0f}%)")

    model.cpu()
    torch.cuda.empty_cache()

def eval_models(models: Sequence[NamedModule], epochs=EPOCHS, parallel=False, lr=1e-3):
    metric_store = Record()
    if parallel:
        devices = [torch.device(f"cuda:{i}") for i in range(len(models))]
        with ThreadPoolExecutor(max_workers=len(models)) as pool:
            futures = [
                pool.submit(eval_model, model, metric_store, dev,
                            ds_train_t, ds_val_t, epochs, lr=lr, position=i)
                for i, (model, dev) in enumerate(zip(models, devices))
            ]
            for f in futures:
                f.result()
        print()  # clean up after tqdm
    else:
        device = torch.device('cuda')
        for model in models:
            eval_model(model, metric_store, device, epochs)
    return metric_store

def test_model(model: nn.Module, device: torch.device):
    metrics = Record()
    model.to(device)
    model.eval()
    loss = nn.CrossEntropyLoss()
    with torch.no_grad():
        for X, y in tqdm(dl_test, leave=False, desc="test"):
            X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
            
            out = model(X)
            l = loss(out, y)
            metrics.add(f"", l.item(), out.detach(), y)
    metrics.compute()
    return (metrics.epoch_store["-loss"][0], metrics.epoch_store["-acc"][0])

def test_models(models: Sequence[NamedModule], device: torch.device):
    out = "Test summary metrics:\n"
    for model in models:
        test_loss, test_acc = test_model(model, device)
        out += f"  {model.name()}: loss={test_loss:.2f}, acc={test_acc*100:.2f}%\n"
    print(out)

device = torch.device('cuda')

## Param Counting

In [ ]:
model2_dummy = CNN(arch=[(1,32,0), (1,64,0)], classes=classes, batch_norm=False) # 1+1
model8_dummy = CNN(arch=[(3,32,0), (3,64,0), (2,128,0)], classes=classes, batch_norm=False) # 3+3+2
model16_dummy = CNN(arch=[(6,32,0), (5,64,0), (5,128,0)], classes=classes, batch_norm=False) # 6+5+5
model32_dummy = CNN(arch=[(8,32,0), (8,64,0), (8,128,0), (8,256,0)], classes=classes, batch_norm=False) # 8+8+8+8

coll = [model2_dummy, model8_dummy, model16_dummy, model32_dummy]

from torchinfo import summary

for i in coll:
    print(summary(i, input_size=(1, 3, 128, 128)))

## Softmax

In [ ]:
linear = SoftmaxRegression(classes)
softmax_metrics = eval_models([linear])

In [ ]:
test_models([linear], device)
plot_comparison(softmax_metrics, "Softmax performance on PathMNIST (128x128)", file_name="softmax_pathmnist")

## Convolutional Neural Network

### Batchnorm off

In [ ]:
# model2_norm_off = CNN(arch=[(1,32,0), (1,64,0)], classes=classes, batch_norm=False) # 1+1
# model8_norm_off = CNN(arch=[(3,32,0), (3,64,0), (2,128,0)], classes=classes, batch_norm=False) # 3+3+2
# model16_norm_off = CNN(arch=[(6,32,0), (5,64,0), (5,128,0)], classes=classes, batch_norm=False) # 6+5+5
# model32_norm_off = CNN(arch=[(8,32,0), (8,64,0), (8,128,0), (8,256,0)], classes=classes, batch_norm=False) # 8+8+8+8

# models_batchnorm_off = [model2_norm_off, model8_norm_off, model16_norm_off, model32_norm_off]
# metrics_batchnorm_off = eval_models(models_batchnorm_off, device, EPOCHS)

In [ ]:
# test_models(models_batchnorm_off, device)
# plot_comparison(metrics_batchnorm_off, "CNN Comparison Normalisation Off", splits=("val","train"), split_subplots=True)

### Batchnorm on

In [ ]:
model2 = CNN(arch=[(1,32,0), (1,64,0)], classes=classes, batch_norm=True) # 1+1
model8 = CNN(arch=[(3,32,0), (3,64,0), (2,128,0)], classes=classes, batch_norm=True) # 3+3+2
model16 = CNN(arch=[(6,32,0), (5,64,0), (5,128,0)], classes=classes, batch_norm=True) # 6+5+5
model32 = CNN(arch=[(8,32,0), (8,64,0), (8,128,0), (8,256,0)], classes=classes, batch_norm=True) # 8+8+8+8

models_batchnorm_on = [model2, model8, model16, model32]
metrics_batchnorm_on = eval_models(models_batchnorm_on, EPOCHS, parallel=True)

In [ ]:
print(metrics_batchnorm_on.epoch_store)

In [ ]:
test_models(models_batchnorm_on, device)
plot_comparison(metrics_batchnorm_on, "CNN Performance Comparison on PathMNIST (128x128)", splits=("val","train"), split_subplots=True, file_name="cnn_comparison_pathmnist")

### Improving Deeper Model

In [ ]:
model16 = CNN(arch=[(6,32,0), (5,64,0), (5,128,0)], classes=classes, batch_norm=True) # 6+5+5
model32 = CNN(arch=[(8,32,0), (8,64,0), (8,128,0), (8,256,0)], classes=classes, batch_norm=True) # 8+8+8+8
model16improved = CNN(arch=[(6,32,2), (5,64,2), (5,128,2)], classes=classes, dropout_rate=0.3, batch_norm=True)
model32improved = CNN(arch=[(8,32,2), (8,64,2), (8,128,2), (8,256,2)], classes=classes, dropout_rate=0.3, batch_norm=True)

models_improvement_comparison = [model16, model32, model16improved, model32improved]
metrics_improvement_comparison = eval_models(models_improvement_comparison, EPOCHS, parallel=True, lr=1e-4)

In [ ]:
def trim_record(record, max_epochs=30):
    trimmed = Record()
    for key, values in record.epoch_store.items():
        trimmed.epoch_store[key] = values[:max_epochs]
    return trimmed

metrics_trimmed = trim_record(metrics_improvement_comparison, 30)

In [ ]:
test_models(models_improvement_comparison, device)
plot_comparison(metrics_trimmed, "CNN-I Performance Comparison on PathMNIST (128x128)", splits=("val","train"), split_subplots=True, file_name="cnn_comparison_improved_pathmnist")

### Learning Rate Comparison

In [ ]:
metric_learning_rates = Record()

learning_rates = [1e-4, 1e-3, 1e-2, 1.0]
for rate in learning_rates:
    model = CNN(arch=[(1,32,0), (1,64,0)], classes=classes, batch_norm=True).set_name(f"lr-{rate}")
    eval_model(model, metric_learning_rates, ds_train_t, ds_val_t, device, EPOCHS, lr=rate)

In [ ]:
plot_comparison(metric_learning_rates, "CNN Learning Rate Comparison", splits=("val","train"), split_subplots=True)

In [ ]:
# X, y = next(iter(dl_test))
# m = nn.Softmax(dim=1)
# # If model on gpu
# # X_gpu = X.to(device, non_blocking=True)
# # y_hat_probs = m(model(X_gpu)).cpu()

# y_hat_probs = m(model32improved(X))

# y_hat = y_hat_probs.argmax(axis=1)

# img = torch.einsum("nchw->nhwc",X)
# fig = px.imshow(img, binary_string=True, facet_col=0, facet_col_wrap=8, height=2000, facet_col_spacing=0.01, facet_row_spacing=0.01)

# for i, label in enumerate(y_hat):
#     text = f"{label_lookup[str(label.item())]} ({y_hat_probs[i][label]*100:.2f}%)"
#     if label != y[i]:
#         text = f"<span style='color:red'>{text}</span>"
    
#     item_map[f'{i}'] = text

# fig.for_each_annotation(lambda a: a.update(text=item_map[a.text.split("=")[1]])) 

# fig.show()